# Librerías

In [2]:
import gymnasium as gym # Importa la librería Gymnasium, que se usa para crear y manejar entornos de aprendizaje por refuerzo
import numpy as np # Importa NumPy, una librería para operaciones matemáticas y manejo eficiente de arreglos
from random import randint # Importa la función randint del módulo random para generar números enteros aleatorios
# Configurar NumPy para no usar notación científica al imprimir
np.set_printoptions(suppress=True)

# Ambiente

[Pendulum](https://gymnasium.farama.org/environments/classic_control/pendulum/)

![Pendulum](https://gymnasium.farama.org/_images/pendulum.gif)

**Péndulo**

El problema del péndulo invertido oscilando hacia arriba se basa en el problema clásico de la teoría de control. El sistema consiste en un péndulo sujeto por un extremo a un punto fijo, mientras que el otro extremo queda libre. El péndulo parte de una posición aleatoria y el objetivo es aplicar un par de torsión en el extremo libre para que oscile hasta una posición vertical, con su centro de gravedad justo encima del punto fijo.

El diagrama que aparece a continuación especifica el sistema de coordenadas utilizado para la implementación de las ecuaciones dinámicas del péndulo.

![Pendulum_2](https://gymnasium.farama.org/_images/pendulum.png)	

Aquí está bien redactado y consistente:

* $x, y$: coordenadas cartesianas del extremo del péndulo (en metros).
* $\theta$: ángulo medido en radianes.
* $\tau$: torque aplicado (en N·m), definido como positivo en sentido antihorario.
.N m

In [9]:
env= gym.make("Pendulum-v1")
env

<TimeLimit<OrderEnforcing<PassiveEnvChecker<PendulumEnv<Pendulum-v1>>>>>

**Objetivo del MDP**

- Llevar el péndulo a la posición vertical superior ($\theta = 0$) y mantenerlo estable.
- Requiere control estratégico: el agente puede necesitar balancear el péndulo (moverlo hacia ambos lados) para acumular energía antes de estabilizarlo arriba.
- La recompensa típica penaliza desviaciones del ángulo vertical, altas velocidades angulares y el uso excesivo de torque.
- El episodio termina cuando:
    1. Se alcanza un número máximo de iteraciones, o
    2. El péndulo se mantiene suficientemente cerca de la posición vertical (dependiendo de la implementación).

**Estados (State Space)**


La observación es un `ndarray` con forma `(3,)` que representa las coordenadas $x$-$y$ del extremo libre del péndulo y su velocidad angular.

| Número | Observación        | Mínimo | Máximo |
| ------ | ------------------ | ------ | ------ |
| 0      | $x = \cos(\theta)$ | -1.0   | 1.0    |
| 1      | $y = \sin(\theta)$ | -1.0   | 1.0    |
| 2      | Velocidad angular  | -8.0   | 8.0    |


In [12]:
low = env.observation_space.low
high = env.observation_space.high

# Extraemos directamente cada componente
x_min, y_min, vel_min = low
x_max, y_max, vel_max = high

print("x (cos(theta))     ∈ [{}, {}]".format(x_min, x_max))
print("y (sin(theta))     ∈ [{}, {}]".format(y_min, y_max))
print("velocidad angular ∈ [{}, {}]".format(vel_min, vel_max))

x (cos(theta))     ∈ [-1.0, 1.0]
y (sin(theta))     ∈ [-1.0, 1.0]
velocidad angular ∈ [-8.0, 8.0]


In [13]:
# Imprime la descripción completa del espacio de observaciones del entorno
print(env.observation_space) # Esto nos dice el tipo de espacio (Box, Discrete, etc.), los valores mínimos y máximos y la forma del estado.

# Imprime los valores mínimos posibles para cada dimensión del estado
# Por ejemplo, en MountainCar-v0: [posición mínima, velocidad mínima]
print("Low:", env.observation_space.low)

# Imprime los valores máximos posibles para cada dimensión del estado
# Por ejemplo, en MountainCar-v0: [posición máxima, velocidad máxima]
print("High:", env.observation_space.high)

# Imprime la forma (shape) del espacio de observaciones
# Esto indica cuántas dimensiones tiene cada estado
# Por ejemplo, (2,) significa que el estado tiene 2 valores (posición y velocidad)
print("Shape:", env.observation_space.shape)

# Imprime el tipo de datos que usa el espacio de observaciones
# Por ejemplo, float32 significa que cada dimensión del estado es un número decimal
print("Dtype:", env.observation_space.dtype)

Box([-1. -1. -8.], [1. 1. 8.], (3,), float32)
Low: [-1. -1. -8.]
High: [1. 1. 8.]
Shape: (3,)
Dtype: float32




**Acciones (Action Space) – Valores Continuos**

El espacio de acciones es continuo y representa el torque aplicado al péndulo:

* $\tau \in [-2.0, 2.0]$

  * Valores negativos → torque en sentido horario
  * Valores positivos → torque en sentido ala” como antes:

| Acción | Descripción                | Mínimo | Máximo |
| ------ | -------------------------- | ------ | ------ |
| $\tau$ | Torque aplicado al péndulo | -2.0   | 2.0    |


Imprime la descripción completa del espacio de **acciones** del entorno.

Esto nos dice el tipo de espacio (Discrete, Box, etc.), los valores mínimos y máximos

In [16]:
print(env.action_space)

Box(-2.0, 2.0, (1,), float32)


Verificamos si el espacio de acciones es discreto o continuo

In [18]:
# Espacios discretos representan acciones como enteros: 0, 1, 2, ...
if isinstance(env.action_space, gym.spaces.Discrete):
    # Imprime el número total de acciones posibles
    print("Número de acciones discretas:", env.action_space.n)
    # Imprime la lista de acciones posibles (enteros)
    print("Acciones posibles:", list(range(env.action_space.n)))
else:
    # Para espacios continuos (Box), imprimimos los rangos mínimos y máximos
    # Esto representa el valor mínimo y máximo que se puede aplicar para cada acción
    print("Acciones continuas - mínimo:", env.action_space.low)
    print("Acciones continuas - máximo:", env.action_space.high)
    # También podemos mostrar la forma y tipo de datos
    print("Shape del espacio de acciones:", env.action_space.shape)
    print("Tipo de datos:", env.action_space.dtype)

Acciones continuas - mínimo: [-2.]
Acciones continuas - máximo: [2.]
Shape del espacio de acciones: (1,)
Tipo de datos: float32


La función  <code> discretizar </code> normaliza el estado 'valor' usando los límites del espacio de observaciones
$$\frac{\textit{valor} - \textit{mínimo}}{\textit{máximo} - \textit{mínimo}}$$
- Cada dimensión queda entre $0$ y $1$.

In [20]:
def discretizar(valor):
    aux = ((valor - env.observation_space.low) / 
           (env.observation_space.high - env.observation_space.low)) * 19
    # Multiplicamos por 20 para crear 20 intervalos discretos por dimensión

    # Convertimos los valores a enteros (truncando decimales)
    aux = aux.astype(np.int32)

    # Devolvemos una tupla de enteros
    # Esto permite usarla como índice en una tabla Q o diccionario
    return tuple(aux)

La discretización en <code>Pendulum-v1</code> se utiliza porque **los estados del entorno son continuos** 
- $x=\cos(\theta)$
- $y=\sin(\theta)$
- Velocidad angular
  
lo que hace imposible usar una Q-table directamente.

Al discretizar, se convierten estos valores continuos en un número finito de “intervalos” o estados. Esto permite representar el problema con una tabla finita y aplicar Q-learning de forma práctica.

Además, reduce la complejidad del problema y facilita el aprendizaje del agente, ya que agrupa estados similares. Sin embargo, existe un equilibrio: usar pocos intervalos simplifica el aprendizaje pero pierde precisión, mientras que usar muchos mejora la precisión pero incrementa el tiempo de entrenamiento y el tamaño de la Q-table.


**Estado Inicial**

* El ángulo del péndulo ($\theta$) se asigna de forma aleatoria en el rango $[-\pi, \pi]$.
* La velocidad angular inicial ($\dot{\theta}$) se asigna típicamente a $0$ o a un valor pequeño cercano a cero (dependiendo de la implementación).
* En la observación, esto se representa como:

  * $x = \cos(\theta)$
  * $y = \sin(\theta)$


**Valor máximo al realizar discretización**

In [25]:
estado_inicial_max = np.array([x_max,y_max,vel_max]) #env.reset()[0]
estado_inicial_max

array([1., 1., 8.], dtype=float32)

In [26]:
discretizar(estado_inicial_max) #Estado inicial

(19, 19, 19)

**Valor aleatorio al realizar discretización**

In [28]:
estado_inicial =  env.reset()[0]
estado_inicial

array([ 0.35215104, -0.9359432 , -0.35912487], dtype=float32)

In [29]:
discretizar(estado_inicial) #Estado inicial

(12, 0, 9)

**Valor mínimo al realizar discretización**

In [31]:
estado_inicial_min = np.array([x_min,y_min,vel_min]) #env.reset()[0]
estado_inicial_min

array([-1., -1., -8.], dtype=float32)

In [32]:
discretizar(estado_inicial_min) #Estado inicial

(0, 0, 0)

# Modelo

Crear la Q-table para un entorno con **3 dimensiones discretizadas** ($x$, $y$ y velocidad angular) y **acciones posibles**

* Cada celda $q\_table[i, j, k, a]$ representa la estimación de la recompensa esperada para el estado discreto $(i, j, k)$ al tomar la acción $a$


Si quieres hacerlo más explícito:

* $i$ → índice para $x = \cos(\theta)$
* $j$ → índice para $y = \sin(\theta)$
* $k$ → índice para la velocidad angular
* $a$ → acción (torque discretizado)

Creamos la Q-table con valores iniciales aleatorios entre -$1$ y $1$

Dimensiones: $[20, 20, 20, 5]$
- $20$ divisiones para x = cos(theta)
- $20$ divisiones para y = sin(theta)
- $20$ divisiones para la velocidad angular
- $5$ acciones posibles (torques discretizados)

In [36]:
q_table = np.random.uniform(low=-1, high=1, size=[20,20,20,5])

In [37]:
len(q_table)

20

**¿Por qué usamos $20$ pero en discretización se multiplica por $19$?**

Porque los índices en Python empiezan en $0$.

Si tenemos $20$ intervalos, los índices válidos son:

$0, 1, 2, \ldots , 19$   → en total $20$ valores

Cuando discretizamos, normalmente hacemos algo como:
indice = (valor_normalizado * (n_bins - 1))

En este caso:
n_bins = $20$  → entonces usamos $(20 - 1) = 19$

Esto asegura que:
- El valor mínimo caiga en el índice $0$
- El valor máximo caiga en el índice $19$

Ejemplo:
valor_normalizado ∈ $[0, 1]$

$0 \cdot 19 = 0$


$1 \cdot 19 = 19$

Así evitamos que el índice se salga del rango de la Q-table.

**Función que convierte una acción DISCRETA (índice) en un valor CONTINUO de torque para el entorno Pendulum**

In [40]:
def continous(indice):

  # Si el índice es 0 → torque muy negativo (giro fuerte horario)
  if indice == 0:
    return np.array([-1.99], dtype=np.float32)

  # Si el índice es 1 → torque negativo moderado
  elif indice == 1:
    return np.array([-1.0], dtype=np.float32)

  # Si el índice es 2 → sin torque (no aplica fuerza)
  elif indice == 2:
    return np.array([0.0], dtype=np.float32)

  # Si el índice es 3 → torque positivo moderado
  elif indice == 3:
    return np.array([1.0], dtype=np.float32)

  # Cualquier otro índice (en este caso 4) → torque muy positivo (giro fuerte antihorario)
  else:
    return np.array([1.99], dtype=np.float32)

In [41]:
def continous(indice):
    if indice == 0:
        return np.array([-2.0], dtype=np.float32)
    elif indice == 1:
        return np.array([-1.0], dtype=np.float32)
    elif indice == 2:
        return np.array([0.0], dtype=np.float32)
    elif indice == 3:
        return np.array([1.0], dtype=np.float32)
    elif indice == 4:
        return np.array([2.0], dtype=np.float32)

IDEA CLAVE:

- El agente trabaja con acciones DISCRETAS: ${0,1,2,3,4}$
- Pero el entorno necesita acciones CONTINUAS: $τ ∈ [-2, 2]$
- Esta función hace la conversión (discretización inversa)

Nota:

Usamos -$1.99$ y $1.99$ en lugar de -$2$ y $2$ para evitar posibles problemas en los límites del entorno.

## Entrenamiento

## Parámetros de Q-learning

In [45]:
alfa = 0.1        # Tasa de aprendizaje: qué tanto se actualiza la Q-table en cada iteración
gamma = 0.95      # Factor de descuento: qué tanto importan las recompensas futuras
episodios = 5000  # Número total de episodios de entrenamiento
epsilon = 2       # Parámetro de exploración (controla aleatoriedad en acciones)
lista_recompenzas = [] # Lista para guardar la recompensa total obtenida en cada episodio

## Bucle principal de entrenamiento

In [47]:
# Reiniciamos el entorno y discretizamos el estado inicial
estado = discretizar(env.reset()[0])

# Política ε-greedy: 
# randint(0,10) -> {0,1,2,3,4,5,6,7,8,9,10} 
if randint(0,10) > epsilon: #{3,4,5,6,7,8,9,10}  8/11=72.73%  
    accion = np.argmax(q_table[estado])  # Con probabilidad alta elegimos la mejor acción conocida (explotación)
else: #{0,1,2} 3/11 = 27.27%
    accion = randint(0,4)  # Con probabilidad baja elegimos una acción aleatoria (exploración)

# Ejecutamos la acción en el entorno
# continous(accion) convierte acción discreta a continua (ej. Pendulum)
nuevo_estado, recompensa, f1, f2, info = env.step(continous(accion))
print(f"nuevo_estado: {nuevo_estado}, recompensa: {recompensa}, f1: {f1}, f2: {f2}, info: {info}")

nuevo_estado: [-0.2555818   0.96678746  0.15191488], recompensa: -3.351426601135752, f1: False, f2: False, info: {}


In [48]:
for episodio in range(episodios):

    # Reiniciamos el entorno y discretizamos el estado inicial
    estado = discretizar(env.reset()[0])

    # Variables de control del episodio
    final = False
    f1 = False
    f2 = False
    
    recompensa_total = 0

    # ==============================
    # INTERACCIÓN CON EL ENTORNO
    # ==============================
    while not final:

        # Política ε-greedy
        valor_random = randint(0,10)
        #print(f"\nEpisodio: {episodio} | Valor random: {valor_random}")

        if valor_random > epsilon:
            accion = np.argmax(q_table[estado])
            tipo = "EXPLOTACIÓN"
        else:
            accion = randint(0,4)
            tipo = "EXPLORACIÓN"

        #print(f"Tipo: {tipo}, Acción: {accion}, Estado actual: {estado}")

        # Ejecutamos la acción
        nuevo_estado, recompensa, f1, f2, info = env.step(continous(accion))

        #print(f"nuevo_estado: {nuevo_estado}, recompensa: {recompensa}, f1: {f1}, f2: {f2}, info: {info}")

        # ==============================
        # ACTUALIZACIÓN Q-LEARNING
        # ==============================
        estado_discreto_nuevo = discretizar(nuevo_estado)

        valor_actual = q_table[estado][accion]
        max_futuro = np.max(q_table[estado_discreto_nuevo])

        #print(f"Q actual: {valor_actual}, max Q futuro: {max_futuro}")

        # Fórmula Q-learning
        q_table[estado][accion] = (
            valor_actual +
            alfa * (recompensa + gamma * max_futuro - valor_actual)
        )

        #print(f"Q actualizado: {q_table[estado][accion]}")

        # Actualizamos estado
        estado = estado_discreto_nuevo

        # Acumulamos recompensa
        recompensa_total += recompensa

        # Verificamos fin
        final = f1 or f2

        #print(f"Recompensa acumulada: {recompensa_total}, Final: {final}")
        #print("--------------------------------------------------")

    # Guardamos recompensa del episodio
    lista_recompenzas.append(recompensa_total)

    # Print cada 100 episodios
    if (episodio + 1) % 100 == 0:
        print("Episodio: " + str(episodio) +
              " | Recompensa promedio: " + str(np.mean(lista_recompenzas)))

# Cerramos entorno
env.close()

Episodio: 99 | Recompensa promedio: -1217.7160050783684
Episodio: 199 | Recompensa promedio: -1235.299403486592
Episodio: 299 | Recompensa promedio: -1226.8467029721667
Episodio: 399 | Recompensa promedio: -1212.1334300375454
Episodio: 499 | Recompensa promedio: -1204.184217527274
Episodio: 599 | Recompensa promedio: -1203.4556432859706
Episodio: 699 | Recompensa promedio: -1190.8068309159403
Episodio: 799 | Recompensa promedio: -1180.2357091338395
Episodio: 899 | Recompensa promedio: -1171.4741041689413
Episodio: 999 | Recompensa promedio: -1164.984948808534
Episodio: 1099 | Recompensa promedio: -1157.84518394867
Episodio: 1199 | Recompensa promedio: -1154.2000186619018
Episodio: 1299 | Recompensa promedio: -1148.1750437823782
Episodio: 1399 | Recompensa promedio: -1136.8316684325612
Episodio: 1499 | Recompensa promedio: -1127.277501168145
Episodio: 1599 | Recompensa promedio: -1116.9151208885764
Episodio: 1699 | Recompensa promedio: -1106.4825200368423
Episodio: 1799 | Recompensa pro

Crear el entorno de Pendulum-v1
- <code> render_mode="human" </code> permite ver la animación en pantalla 

In [88]:
env = gym.make("Pendulum-v1", render_mode="human")

In [51]:
print("Estado:", estado, "Tipo:", type(estado))
print("q_table shape:", q_table.shape)
print("q_table[estado]:", q_table[estado])

Estado: (18, 9, 8) Tipo: <class 'tuple'>
q_table shape: (20, 20, 20, 5)
q_table[estado]: [-0.51610517 -0.49204593 -0.58958286 -0.40228816 -0.3412378 ]


In [52]:
print(f"Estado: {estado}, len: {len(estado)}")

Estado: (18, 9, 8), len: 3


In [90]:
# Reinicia el entorno y obtiene el estado inicial (continuo)
# Luego se discretiza para poder usar la Q-table
estado = discretizar(env.reset()[0])

# Variables de control del episodio
final = False   # Indica si el episodio terminó
f1 = False      # Bandera de éxito (llegó a la meta)
f2 = False      # Bandera de fallo (se terminó el tiempo)


# Mientras el episodio no termine
while not final:
    
    # Elige la mejor acción según la Q-table (explotación)
    accion = np.argmax(q_table[estado])
      
    # Ejecuta la acción en el entorno
    # Devuelve el nuevo estado, la recompensa y si terminó el episodio
    nuevo_estado, recompensa, f1, f2, info = env.step(continous(accion))

    # Convierte el nuevo estado continuo a discreto
    estado = discretizar(nuevo_estado)
  
    # Verifica si el episodio terminó (éxito o fallo)
    final = f1 or f2

# Cierra el entorno
env.close()